In [1]:
import numpy as np
import axelrod
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import random
from copy import deepcopy
from collections import Counter
from math import exp


In [2]:
PARAM_KEYS = [
    'D_b', 'D_c',
    'C_a', 'C_b',
    'd_threshold', 'c_threshold',
]

BOUNDS = [
    (1,  49),   # D_b
    (1,  60),   # D_c
    (25, 74),   # C_a
    (25, 99),   # C_b
    (0.1, 0.8), # d_threshold
    (0.2, 0.9), # c_threshold
]

YOUR_BASELINE = [
    25, 50,   # D_b, D_c
    35, 75,   # C_a, C_b
    0.4, 0.6, # thresholds
]

In [3]:
def decode(individual):
    """Convert flat list back to named params dict."""
    return dict(zip(PARAM_KEYS, individual))


def repair(individual):
    ind = individual.copy()

    # Clip to bounds
    for i, (lo, hi) in enumerate(BOUNDS):
        ind[i] = float(np.clip(ind[i], lo, hi))

    # D_b <= D_c
    ind[1] = max(ind[0], ind[1])

    # C_a <= C_b
    ind[3] = max(ind[2], ind[3])

    return ind

In [4]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def sigmoid(x, center, scale=10):
        return 1 / (1 + exp(-scale * (x - center)))
    
    @staticmethod
    def fuzzy_gate(mu_D, d_thresh, mu_C, c_thresh):
        d_condition = FuzzyMethods.sigmoid(mu_D, center=d_thresh)
        c_condition = 1 - FuzzyMethods.sigmoid(mu_C, center=c_thresh)

        w1 = d_condition * c_condition
        w2 = 1 - w1

        z1 = 1
        z2 = 0

        z = (w1 * z1 + w2 * z2) / (w1 + w2 + 1e-6)
        return z
    
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100
    

In [5]:
def build_player(params):
    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')
    

    _cooperation.automf(names=["low", "medium", "high"])
    _adaptivity.automf(names=["no", "yes"])
    _forgiveness.automf(names=["low", "medium", "high"])
    _forgiveness['low'] = fuzz.gaussmf(_forgiveness.universe, 0, 25)
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe, [25, 50, 75])
    _stochastic.automf(names=["none", "sometimes", "always"])

    # Resulting strategy MFs — also being optimized
    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [
        0,
        params['D_b'],
        params['D_c']
    ])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [
        params['C_a'],
        params['C_b'],
        100
    ])

    # Rebuild rules using the fresh variables above
    rule1 = ctrl.Rule(
        _cooperation['high'] & _adaptivity['no'] & (_forgiveness['medium'] | _forgiveness['high']),
        _resulting_strategy['D']
    )
    rule2 = ctrl.Rule(
        _forgiveness['low'] & _cooperation['high'],
        _resulting_strategy['C']
    )
    rule3 = ctrl.Rule(
        _stochastic['always'] | _adaptivity['no'],
        _resulting_strategy['D']
    )
    rule4 = ctrl.Rule(
        _cooperation['low'] | (_cooperation['medium'] & _forgiveness['low']),
        _resulting_strategy['D']
    )
    rule5 = ctrl.Rule(
        _cooperation['medium'] & _forgiveness['medium'] & _adaptivity['yes'],
        _resulting_strategy['C']
    )

    strategy_ctrl  = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
    _chosen_strategy = ctrl.ControlSystemSimulation(strategy_ctrl)

    # Build the player class dynamically, capturing everything in closure
    class OptimizedFuzzy(Player):

        # Override class-level FIS components with the fresh ones
        cooperation = _cooperation
        adaptivity = _adaptivity
        stochastic = _stochastic
        forgiveness = _forgiveness
        resulting_strategy = _resulting_strategy
        chosen_strategy = _chosen_strategy

        d_thresh = params['d_threshold']
        c_thresh = params['c_threshold']

        # Reset state so trials don't bleed into each other
        first_time = True
        h = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: axelrod.Player) -> Action:

            if len(self.history) == 0 or D not in opponent.history:
                return C

            coop  = FuzzyMethods.calc_cooperation(self, opponent)
            adap  = FuzzyMethods.calc_adaptivity(self, opponent)
            forg  = FuzzyMethods.calc_forgiveness(self, opponent)
            stoch = FuzzyMethods.calc_stochastic(self, opponent)

            self.chosen_strategy.input['cooperation'] = coop
            self.chosen_strategy.input['adaptivity']  = adap
            self.chosen_strategy.input['forgiveness'] = forg
            self.chosen_strategy.input['stochastic']  = stoch

            try:
                self.chosen_strategy.compute()
                output_val = self.chosen_strategy.output['resulting_strategy']
            except KeyError:
                # No rules fired — default to cooperate
                return C
            except Exception:
                return C

            d_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['D'].mf,
                output_val
            )
            c_membership = fuzz.interp_membership(
                self.resulting_strategy.universe,
                self.resulting_strategy['C'].mf,
                output_val
            )

            output = FuzzyMethods.fuzzy_gate(d_membership, self.d_thresh, c_membership, self.c_thresh)

            if(output > 0.5):
                return D
            
            return C

    return OptimizedFuzzy()

In [6]:
def build_player_from_individual(individual):
    ind = repair(individual)
    p   = decode(ind)
    params = {
        'D_a': 0,             'D_b': int(p['D_b']), 'D_c': int(p['D_c']),
        'C_a': int(p['C_a']), 'C_b': int(p['C_b']), 'C_c': 99,
        'd_threshold': p['d_threshold'],
        'c_threshold': p['c_threshold'],
    }
    return build_player(params)

In [7]:
def evaluate(individual):
    try:
        fuzzy_player = build_player_from_individual(individual)
        opponents    = [s() for s in axelrod.stewart_plotkin_strategies]
        results      = axelrod.Tournament([fuzzy_player] + opponents, turns=200, repetitions=3).play(progress_bar=False)
        return np.mean(results.normalised_scores[0])
    except Exception as e:
        print(f"  [evaluate ERROR] {type(e).__name__}: {e}")
        return 0.0

In [8]:
# Reuse PARAM_KEYS, BOUNDS, YOUR_BASELINE, decode, repair, build_player, evaluate
# from the GA file — put them in a shared shared_utils.py and import from there


# ─────────────────────────────────────────────
#  PARTICLE SWARM OPTIMIZATION
# ─────────────────────────────────────────────
#
#  Each particle has:
#    position  — current parameter values (the solution)
#    velocity  — how fast/which direction it's moving
#    pbest     — best position this particle personally found
#    gbest     — best position ANY particle found (shared)
#
#  Update rules each iteration:
#    velocity = w * velocity
#             + c1 * r1 * (pbest - position)   ← pull toward personal best
#             + c2 * r2 * (gbest - position)   ← pull toward global best
#    position = position + velocity


class Particle:

    def __init__(self, position):
        self.position  = np.array(position, dtype=float)
        self.velocity  = np.array([
            np.random.uniform(-(hi - lo) * 0.1, (hi - lo) * 0.1)
            for lo, hi in BOUNDS
        ])
        self.pbest          = self.position.copy()
        self.pbest_score    = -np.inf

    def update_velocity(self, gbest, w, c1, c2):
        r1 = np.random.uniform(0, 1, size=len(self.position))
        r2 = np.random.uniform(0, 1, size=len(self.position))

        cognitive = c1 * r1 * (self.pbest   - self.position)
        social    = c2 * r2 * (gbest        - self.position)

        self.velocity = w * self.velocity + cognitive + social

        # Clamp velocity to 20% of range to prevent explosion
        for i, (lo, hi) in enumerate(BOUNDS):
            max_v            = (hi - lo) * 0.2
            self.velocity[i] = np.clip(self.velocity[i], -max_v, max_v)

    def update_position(self):
        self.position = repair(list(self.position + self.velocity))
        self.position = np.array(self.position)


def run_pso(
    n_particles  = 30,
    n_iterations = 50,
    w            = 0.7,   # inertia weight — how much old velocity is kept
    c1           = 1.5,   # cognitive coefficient — trust in personal best
    c2           = 1.5,   # social coefficient — trust in global best
    w_decay      = 0.99,  # slowly reduce inertia to shift from explore to exploit
):
    print("Initialising swarm...")

    # Initialise particles — seed one with your baseline
    particles    = [Particle(YOUR_BASELINE.copy())]
    particles   += [Particle([np.random.uniform(lo, hi) for lo, hi in BOUNDS])
                    for _ in range(n_particles - 1)]

    gbest        = particles[0].position.copy()
    gbest_score  = -np.inf

    for iteration in range(n_iterations):

        for particle in particles:

            score = evaluate(list(particle.position))

            # Update personal best
            if score > particle.pbest_score:
                particle.pbest_score = score
                particle.pbest       = particle.position.copy()

            # Update global best
            if score > gbest_score:
                gbest_score = score
                gbest       = particle.position.copy()

        # Update velocities and positions
        for particle in particles:
            particle.update_velocity(gbest, w, c1, c2)
            particle.update_position()

        # Decay inertia — more exploitation as iterations progress
        w *= w_decay

        scores = [p.pbest_score for p in particles]
        print(f"Iter {iteration+1:>4}/{n_iterations} | GBest: {gbest_score:.4f} | Swarm avg pbest: {np.mean(scores):.4f} | w: {w:.4f}")

    print(f"\n=== PSO COMPLETE ===")
    print(f"Best score: {gbest_score:.4f}")
    print(f"Best params: {decode(list(gbest))}")
    return gbest, gbest_score


if __name__ == "__main__":
    best_pos, best_score = run_pso()

Initialising swarm...
Iter    1/50 | GBest: 2.7807 | Swarm avg pbest: 2.6851 | w: 0.6930
Iter    2/50 | GBest: 2.7813 | Swarm avg pbest: 2.7310 | w: 0.6861
Iter    3/50 | GBest: 2.7913 | Swarm avg pbest: 2.7473 | w: 0.6792
Iter    4/50 | GBest: 2.7913 | Swarm avg pbest: 2.7542 | w: 0.6724
Iter    5/50 | GBest: 2.7913 | Swarm avg pbest: 2.7593 | w: 0.6657
Iter    6/50 | GBest: 2.7913 | Swarm avg pbest: 2.7610 | w: 0.6590
Iter    7/50 | GBest: 2.7913 | Swarm avg pbest: 2.7625 | w: 0.6524
Iter    8/50 | GBest: 2.7913 | Swarm avg pbest: 2.7634 | w: 0.6459
Iter    9/50 | GBest: 2.7956 | Swarm avg pbest: 2.7649 | w: 0.6395
Iter   10/50 | GBest: 2.7956 | Swarm avg pbest: 2.7677 | w: 0.6331
Iter   11/50 | GBest: 2.7956 | Swarm avg pbest: 2.7694 | w: 0.6267
Iter   12/50 | GBest: 2.7956 | Swarm avg pbest: 2.7703 | w: 0.6205
Iter   13/50 | GBest: 2.7956 | Swarm avg pbest: 2.7714 | w: 0.6143
Iter   14/50 | GBest: 2.7956 | Swarm avg pbest: 2.7717 | w: 0.6081
Iter   15/50 | GBest: 2.7956 | Swarm avg